In [19]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor

# 1. Load dataset
df = pd.read_excel(r"C:\Capstone\Updated_capstone.xlsx")

# 2. Remove unwanted unnamed columns
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# 3. Check missing values
print("Missing Values:")
print(df.isnull().sum())

# 4. Encode categorical columns
label_cols = ['Part_Name', 'Category', 'Supplier', 'Delivery_Status', 'BRAND', 'VEHICLE_TYPE', 'DEMAND', 'Season']

label_encoders = {} 

for col in label_cols:
    if col in df.columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        label_encoders[col] = le

# 5. Select features and target
X = df[['Month',
        'CURRENT_STOCK',
        'REORDER_LEVEL',
        'Supplier_Lead_Time_Days',
        'Festival_Season',
        'Weekend',
        'Category',
        'BRAND',
        'Season']]

y = df['Quantity_Sold']

# 6. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 7. Train models

# Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

# Random Forest
rf = RandomForestRegressor(
    n_estimators=500,
    random_state=42
)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

# XGBoost
xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)

# 8. Evaluation function
def evaluate_model(model_name, y_test, y_pred):
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    return {
        'Model': model_name,
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2
    }

# 9. Results table
results_df = pd.DataFrame([
    evaluate_model("Linear Regression", y_test, lr_pred),
    evaluate_model("Random Forest", y_test, rf_pred),
    evaluate_model("XGBoost", y_test, xgb_pred)
])

print("\nModel Comparison:")
print(results_df)

# 10. Final prediction using best model - XGBoost
df['Predicted_Demand'] = xgb.predict(X)

# 11. Prediction error
df['Prediction_Error'] = df['Quantity_Sold'] - df['Predicted_Demand']

# 12. Round prediction columns
df['Predicted_Demand'] = df['Predicted_Demand'].round(2)
df['Prediction_Error'] = df['Prediction_Error'].round(2)

# 13. Check final columns
print("\nFinal Columns:")
print(df.columns)

# 14. Save final dataset for Power BI
df.to_excel("Final_SpareParts_Prediction.xlsx", index=False)
df.to_csv("Final_SpareParts_Prediction.csv", index=False)
print("\nFinal file saved successfully: Final_SpareParts_Prediction.xlsx")

Missing Values:
Transaction_ID             0
Date                       0
Month                      0
Year                       0
Part_ID                    0
Part_Name                  0
Category                   0
Supplier                   0
Quantity_Sold              0
Unit_Cost_INR              0
Unit_Retail_Price_INR      0
Total_Revenue_INR          0
Total_Cost_INR             0
Net_Profit_INR             0
Supplier_Lead_Time_Days    0
Delivery_Status            0
BRAND                      0
CURRENT_STOCK              0
REORDER_LEVEL              0
VEHICLE_TYPE               0
DEMAND                     0
DayOfWeek                  0
Revenue                    0
Weekend                    0
Season                     0
Festival_Season            0
dtype: int64

Model Comparison:
               Model       MAE      RMSE        R2
0  Linear Regression  3.917003  4.680613  0.974307
1      Random Forest  4.903583  6.301160  0.953436
2            XGBoost  4.693629  5.622830  0.9

In [20]:
# Import required libraries
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# MODEL 2: Predict Demand Category

X_demand = df[['Month',
               'CURRENT_STOCK',
               'REORDER_LEVEL',
               'Supplier_Lead_Time_Days',
               'Festival_Season',
               'Weekend',
               'Category',
               'BRAND',
               'Season']]

y_demand = df['DEMAND']

X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_demand,
    y_demand,
    test_size=0.2,
    random_state=42
)

# Train Random Forest Classifier
demand_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

demand_model.fit(X_train_d, y_train_d)

# Prediction
demand_pred = demand_model.predict(X_test_d)

# Evaluation
print(" Demand Category Classification ")
print("Accuracy :", accuracy_score(y_test_d, demand_pred))

print("\nClassification Report")
print(classification_report(y_test_d, demand_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test_d, demand_pred))

# Predict for entire dataset
df['Predicted_Demand_Category'] = demand_model.predict(X_demand)

print("\nFirst 10 Predictions")
print(df[['DEMAND','Predicted_Demand_Category']].head(10))

===== Demand Category Classification =====
Accuracy : 0.8833333333333333

Classification Report
              precision    recall  f1-score   support

           0       1.00      0.85      0.92        26
           1       0.87      0.82      0.84        33
           2       0.85      0.93      0.89        61

    accuracy                           0.88       120
   macro avg       0.91      0.87      0.88       120
weighted avg       0.89      0.88      0.88       120


Confusion Matrix
[[22  0  4]
 [ 0 27  6]
 [ 0  4 57]]

First 10 Predictions
   DEMAND  Predicted_Demand_Category
0       0                          0
1       2                          2
2       1                          1
3       2                          2
4       2                          2
5       0                          0
6       2                          2
7       1                          1
8       2                          2
9       2                          2


In [21]:
# MODEL 3: CLASSIFICATION - Predict Delivery Status
# Fixed Version

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd

# Features
X_delivery = df[['Supplier_Lead_Time_Days',
                 'CURRENT_STOCK',
                 'REORDER_LEVEL',
                 'Month',
                 'Festival_Season',
                 'Weekend',
                 'Category',
                 'BRAND',
                 'Season']]

# Target
y_delivery = df['Delivery_Status']

# Train-test split with stratify to keep class balance
X_train_del, X_test_del, y_train_del, y_test_del = train_test_split(
    X_delivery,
    y_delivery,
    test_size=0.2,
    random_state=42,
    stratify=y_delivery
)

# Balanced Random Forest
delivery_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=6,
    min_samples_split=4,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42
)

# Train model
delivery_model.fit(X_train_del, y_train_del)

# Predict test data
delivery_pred = delivery_model.predict(X_test_del)

# Evaluation
print("===== Delivery Status Classification - Fixed =====")
print("Accuracy:", accuracy_score(y_test_del, delivery_pred))

print("\nClassification Report:")
print(classification_report(y_test_del, delivery_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_del, delivery_pred))

# Predict full dataset
df['Predicted_Delivery_Status'] = delivery_model.predict(X_delivery)

print("\nFirst 10 Predictions:")
print(df[['Delivery_Status', 'Predicted_Delivery_Status']].head(10))

# Save final file
df.to_excel("Final_SpareParts_Complete_ML_Output.xlsx", index=False)
df.to_csv("Final_SpareParts_Complete_ML_Output.csv", index=False)

print("\nFinal file saved successfully: Final_SpareParts_Complete_ML_Output.xlsx")

===== Delivery Status Classification - Fixed =====
Accuracy: 0.7666666666666667

Classification Report:
              precision    recall  f1-score   support

           0       0.09      0.05      0.07        19
           1       0.83      0.90      0.87       101

    accuracy                           0.77       120
   macro avg       0.46      0.48      0.47       120
weighted avg       0.72      0.77      0.74       120


Confusion Matrix:
[[ 1 18]
 [10 91]]

First 10 Predictions:
   Delivery_Status  Predicted_Delivery_Status
0                1                          1
1                1                          1
2                1                          1
3                1                          1
4                1                          1
5                1                          1
6                1                          1
7                1                          1
8                1                          1
9                1                          1

F

In [22]:
df.to_csv("PowerBI_Dashboard_Dataset.csv", index=False)

In [23]:
columns_to_decode = ['Part_Name', 'Supplier', 'Category', 'VEHICLE_TYPE', 'DEMAND', 'BRAND', 'Delivery_Status', 'Season']

for col in columns_to_decode:
    # This safely checks if we have a saved encoder for this column
    if col in df.columns and col in label_encoders:
        df[col] = label_encoders[col].inverse_transform(df[col].astype(int))

df.to_excel("PowerBI_Dashboard_Dataset.xlsx", index=False)
df.to_csv("PowerBI_Dashboard_Dataset.csv", index=False)
print("Power BI dataset created successfully!")

Power BI dataset created successfully!


In [24]:
# Create a copy so we don't break anything else
df_powerbi = df.copy()

# 1. Decode the columns handled by LabelEncoder (only if they are still numeric codes)
columns_to_decode = ['Part_Name', 'Category', 'Supplier', 'Delivery_Status', 'BRAND', 'VEHICLE_TYPE', 'DEMAND', 'Season']

for col in columns_to_decode:
    if col in df_powerbi.columns and col in label_encoders:
        # ONLY decode if the column contains numbers; skip if it's already text words
        if df_powerbi[col].dtype in ['int64', 'int32', 'float64']:
            df_powerbi[col] = label_encoders[col].inverse_transform(df_powerbi[col].astype(int))

# 2. Decode the Predicted columns using the target column's encoder
if 'Predicted_Demand_Category' in df_powerbi.columns and 'DEMAND' in label_encoders:
    if df_powerbi['Predicted_Demand_Category'].dtype in ['int64', 'int32', 'float64']:
        df_powerbi['Predicted_Demand_Category'] = label_encoders['DEMAND'].inverse_transform(df_powerbi['Predicted_Demand_Category'].astype(int))

if 'Predicted_Delivery_Status' in df_powerbi.columns and 'Delivery_Status' in label_encoders:
    if df_powerbi['Predicted_Delivery_Status'].dtype in ['int64', 'int32', 'float64']:
        df_powerbi['Predicted_Delivery_Status'] = label_encoders['Delivery_Status'].inverse_transform(df_powerbi['Predicted_Delivery_Status'].astype(int))

# 3. Map DayOfWeek, Weekend, and Festive Season mappings safely
day_mapping = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday', 4: 'Friday', 5: 'Saturday', 6: 'Sunday'}
weekend_mapping = {0: 'Weekday', 1: 'Weekend'}
festival_mapping = {0: 'No Festival', 1: 'Festival Season'}

if 'DayOfWeek' in df_powerbi.columns and df_powerbi['DayOfWeek'].dtype in ['int64', 'int32', 'float64']:
    df_powerbi['DayOfWeek'] = df_powerbi['DayOfWeek'].map(day_mapping)

if 'Weekend' in df_powerbi.columns and df_powerbi['Weekend'].dtype in ['int64', 'int32', 'float64']:
    df_powerbi['Weekend'] = df_powerbi['Weekend'].map(weekend_mapping)
    
if 'Festival_Season' in df_powerbi.columns and df_powerbi['Festival_Season'].dtype in ['int64', 'int32', 'float64']:
    df_powerbi['Festival_Season'] = df_powerbi['Festival_Season'].map(festival_mapping)

# 4. Save the cleanly decoded dataset
df_powerbi.to_csv("PowerBI_Dashboard_Dataset.csv", index=False)
print("Decoded Power BI dataset successfully saved!")

Decoded Power BI dataset successfully saved!


In [25]:
# Create a copy so we don't break anything else
df_powerbi = df.copy()

# 1. Decode standard columns (with safety checks)
columns_to_decode = ['Part_Name', 'Category', 'Supplier', 'Delivery_Status', 'DEMAND']
for col in columns_to_decode:
    if col in df_powerbi.columns and col in label_encoders:
        if df_powerbi[col].dtype in ['int64', 'int32', 'float64']:
            df_powerbi[col] = label_encoders[col].inverse_transform(df_powerbi[col].astype(int))

# 2. FORCE decode BRAND, VEHICLE_TYPE, and Season
force_decode = ['BRAND', 'VEHICLE_TYPE', 'Season']
for col in force_decode:
    if col in df_powerbi.columns and col in label_encoders:
        try:
            # Convert to integer first in case they are stored as strings of numbers (e.g., '0', '1')
            numeric_series = df_powerbi[col].astype(float).astype(int)
            df_powerbi[col] = label_encoders[col].inverse_transform(numeric_series)
        except ValueError:
            # If they are already actual words (like 'Toyota', 'Summer'), leave them alone
            pass

# 3. Decode the Predicted columns using the target column's encoder
if 'Predicted_Demand_Category' in df_powerbi.columns and 'DEMAND' in label_encoders:
    if df_powerbi['Predicted_Demand_Category'].dtype in ['int64', 'int32', 'float64']:
        df_powerbi['Predicted_Demand_Category'] = label_encoders['DEMAND'].inverse_transform(df_powerbi['Predicted_Demand_Category'].astype(int))

if 'Predicted_Delivery_Status' in df_powerbi.columns and 'Delivery_Status' in label_encoders:
    if df_powerbi['Predicted_Delivery_Status'].dtype in ['int64', 'int32', 'float64']:
        df_powerbi['Predicted_Delivery_Status'] = label_encoders['Delivery_Status'].inverse_transform(df_powerbi['Predicted_Delivery_Status'].astype(int))

# 4. Map DayOfWeek, Weekend, and Festive Season mappings safely
day_mapping = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday', 4: 'Friday', 5: 'Saturday', 6: 'Sunday'}
weekend_mapping = {0: 'Weekday', 1: 'Weekend'}
festival_mapping = {0: 'No Festival', 1: 'Festival Season'}

if 'DayOfWeek' in df_powerbi.columns:
    df_powerbi['DayOfWeek'] = df_powerbi['DayOfWeek'].astype(float).astype(int).map(day_mapping)

if 'Weekend' in df_powerbi.columns:
    df_powerbi['Weekend'] = df_powerbi['Weekend'].astype(float).astype(int).map(weekend_mapping)
    
if 'Festival_Season' in df_powerbi.columns:
    df_powerbi['Festival_Season'] = df_powerbi['Festival_Season'].astype(float).astype(int).map(festival_mapping)

# 5. Save the cleanly decoded dataset
df_powerbi.to_csv("PowerBI_Dashboard_Dataset2.csv", index=False)
print("Decoded Power BI dataset successfully saved!")

Decoded Power BI dataset successfully saved!


In [26]:
# Create a copy so we don't break anything else
df_powerbi = df.copy()

# 1. Manually decode columns using label_encoders if available, otherwise force them
columns_to_decode = ['Part_Name', 'Category', 'Supplier', 'Delivery_Status', 'BRAND', 'VEHICLE_TYPE', 'DEMAND', 'Season']

for col in columns_to_decode:
    if col in df_powerbi.columns:
        if col in label_encoders:
            try:
                # Force conversion from float/int to integer codes, then transform
                numeric_series = df_powerbi[col].astype(float).fillna(0).astype(int)
                df_powerbi[col] = label_encoders[col].inverse_transform(numeric_series)
            except Exception:
                pass

# 2. Decode the Predicted columns using the target column's encoder
if 'Predicted_Demand_Category' in df_powerbi.columns and 'DEMAND' in label_encoders:
    df_powerbi['Predicted_Demand_Category'] = label_encoders['DEMAND'].inverse_transform(df_powerbi['Predicted_Demand_Category'].astype(float).astype(int))

if 'Predicted_Delivery_Status' in df_powerbi.columns and 'Delivery_Status' in label_encoders:
    df_powerbi['Predicted_Delivery_Status'] = label_encoders['Delivery_Status'].inverse_transform(df_powerbi['Predicted_Delivery_Status'].astype(float).astype(int))

# 3. Map DayOfWeek, Weekend, and Festive Season mappings safely
day_mapping = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday', 4: 'Friday', 5: 'Saturday', 6: 'Sunday'}
weekend_mapping = {0: 'Weekday', 1: 'Weekend'}
festival_mapping = {0: 'No Festival', 1: 'Festival Season'}

if 'DayOfWeek' in df_powerbi.columns:
    df_powerbi['DayOfWeek'] = df_powerbi['DayOfWeek'].astype(float).fillna(0).astype(int).map(day_mapping)

if 'Weekend' in df_powerbi.columns:
    df_powerbi['Weekend'] = df_powerbi['Weekend'].astype(float).fillna(0).astype(int).map(weekend_mapping)
    
if 'Festival_Season' in df_powerbi.columns:
    df_powerbi['Festival_Season'] = df_powerbi['Festival_Season'].astype(float).fillna(0).astype(int).map(festival_mapping)

# 4. Save the cleanly decoded dataset
df_powerbi.to_csv("PowerBI_Dashboard_Dataset_Final.csv", index=False)
print("Decoded Power BI dataset successfully saved!")

Decoded Power BI dataset successfully saved!


In [27]:
df.columns

Index(['Transaction_ID', 'Date', 'Month', 'Year', 'Part_ID', 'Part_Name',
       'Category', 'Supplier', 'Quantity_Sold', 'Unit_Cost_INR',
       'Unit_Retail_Price_INR', 'Total_Revenue_INR', 'Total_Cost_INR',
       'Net_Profit_INR', 'Supplier_Lead_Time_Days', 'Delivery_Status', 'BRAND',
       'CURRENT_STOCK', 'REORDER_LEVEL', 'VEHICLE_TYPE', 'DEMAND', 'DayOfWeek',
       'Revenue', 'Weekend', 'Season', 'Festival_Season', 'Predicted_Demand',
       'Prediction_Error', 'Predicted_Demand_Category',
       'Predicted_Delivery_Status'],
      dtype='object')

In [28]:
df_powerbi.columns

Index(['Transaction_ID', 'Date', 'Month', 'Year', 'Part_ID', 'Part_Name',
       'Category', 'Supplier', 'Quantity_Sold', 'Unit_Cost_INR',
       'Unit_Retail_Price_INR', 'Total_Revenue_INR', 'Total_Cost_INR',
       'Net_Profit_INR', 'Supplier_Lead_Time_Days', 'Delivery_Status', 'BRAND',
       'CURRENT_STOCK', 'REORDER_LEVEL', 'VEHICLE_TYPE', 'DEMAND', 'DayOfWeek',
       'Revenue', 'Weekend', 'Season', 'Festival_Season', 'Predicted_Demand',
       'Prediction_Error', 'Predicted_Demand_Category',
       'Predicted_Delivery_Status'],
      dtype='object')

In [29]:

df_powerbi = df.copy()

# 1. Decode standard columns using label_encoders
columns_to_decode = ['Part_Name', 'Category', 'Supplier', 'Delivery_Status', 'DEMAND']
for col in columns_to_decode:
    if col in df_powerbi.columns and 'label_encoders' in globals():
        if col in label_encoders:
            try:
                numeric_series = df_powerbi[col].astype(float).fillna(0).astype(int)
                df_powerbi[col] = label_encoders[col].inverse_transform(numeric_series)
            except Exception:
                pass

# 2. GUARANTEED MANUAL MAPPING FOR BRAND, VEHICLE_TYPE, & SEASON
# (Adjust the text names below if they mismatch your original data values)
brand_mapping = {0: 'Suzuki', 1: 'Honda', 2: 'Hero', 3: 'Bajaj', 4: 'TVS', 5: 'Yamaha', 6:'Aprilia', 7:'others'} 
vehicle_mapping = {0: 'Scooter', 1: 'Bike'}
season_mapping = {0: 'Summer', 1: 'Monsoon', 2: 'Winter', 3: 'Spring'}

if 'BRAND' in df_powerbi.columns:
    df_powerbi['BRAND'] = df_powerbi['BRAND'].astype(float).fillna(0).astype(int).map(brand_mapping).fillna(df_powerbi['BRAND'])

if 'VEHICLE_TYPE' in df_powerbi.columns:
    df_powerbi['VEHICLE_TYPE'] = df_powerbi['VEHICLE_TYPE'].astype(float).fillna(0).astype(int).map(vehicle_mapping).fillna(df_powerbi['VEHICLE_TYPE'])

if 'Season' in df_powerbi.columns:
    df_powerbi['Season'] = df_powerbi['Season'].astype(float).fillna(0).astype(int).map(season_mapping).fillna(df_powerbi['Season'])

# 3. Decode Predicted columns if they exist
if 'Predicted_Demand_Category' in df_powerbi.columns and 'DEMAND' in label_encoders:
    df_powerbi['Predicted_Demand_Category'] = label_encoders['DEMAND'].inverse_transform(df_powerbi['Predicted_Demand_Category'].astype(float).fillna(0).astype(int))

if 'Predicted_Delivery_Status' in df_powerbi.columns and 'Delivery_Status' in label_encoders:
    df_powerbi['Predicted_Delivery_Status'] = label_encoders['Delivery_Status'].inverse_transform(df_powerbi['Predicted_Delivery_Status'].astype(float).fillna(0).astype(int))

# 4. Map DayOfWeek, Weekend, and Festive Season mappings safely
day_mapping = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday', 4: 'Friday', 5: 'Saturday', 6: 'Sunday'}
weekend_mapping = {0: 'Weekday', 1: 'Weekend'}
festival_mapping = {0: 'No Festival', 1: 'Festival Season'}

if 'DayOfWeek' in df_powerbi.columns:
    df_powerbi['DayOfWeek'] = df_powerbi['DayOfWeek'].astype(float).fillna(0).astype(int).map(day_mapping)

if 'Weekend' in df_powerbi.columns:
    df_powerbi['Weekend'] = df_powerbi['Weekend'].astype(float).fillna(0).astype(int).map(weekend_mapping)
    
if 'Festival_Season' in df_powerbi.columns:
    df_powerbi['Festival_Season'] = df_powerbi['Festival_Season'].astype(float).fillna(0).astype(int).map(festival_mapping)

# 5. decoded dataset
df_powerbi.to_csv("PowerBI_Dashboard_Dataset_Final3.csv", index=False)
print("Decoded Power BI dataset successfully saved!")

Decoded Power BI dataset successfully saved!
